# Jurimetria Preditiva — Acórdãos TCU (Saúde & Educação)

Pipeline completo: download → filtro temático → EDA → baseline TF-IDF → LegalBert-pt → comparação → threshold ótimo → LIME

**Execute no Google Colab com GPU T4 (Runtime → Change runtime type → T4 GPU)**

In [ ]:
# Célula 1 — Setup
import os, sys

# Monte o Google Drive para persistência entre sessões
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_DIR = '/content/drive/MyDrive/deep-acordao-tcu'

# Para execução local, use o diretório do repositório:
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

if REPO_DIR not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Instalar dependências (descomente no Colab)
# !pip install -q -r {REPO_DIR}/requirements.txt

print('Repositório:', REPO_DIR)
print('Python:', sys.version)

In [ ]:
# Célula 2 — Configuração
from pathlib import Path

ANOS = list(range(2020, 2025))          # 2020, 2021, 2022, 2023, 2024
CAMPO_BASELINE = 'SUMARIO'              # D-06: baseline usa SUMARIO
CAMPO_TRANSFORMER = 'VOTO'             # D-06: Transformer usa VOTO
RANDOM_STATE = 42                       # D-guardrail: seed fixa

DATA_RAW       = Path(REPO_DIR) / 'data' / 'raw'
DATA_INTERIM   = Path(REPO_DIR) / 'data' / 'interim'
DATA_PROCESSED = Path(REPO_DIR) / 'data' / 'processed'
RESULTADOS     = Path(REPO_DIR) / 'resultados'
FIGURAS        = RESULTADOS / 'figuras'

for d in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, FIGURAS]:
    d.mkdir(parents=True, exist_ok=True)

print('Configuração OK')
print(f'Anos: {ANOS}')

In [ ]:
# Célula 3 — Download dos CSVs reais do TCU
from aquisicao.baixar_csvs import baixar_todos
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

arquivos = baixar_todos(ANOS, data_dir=DATA_RAW)

print(f'\nArquivos disponíveis: {len(arquivos)}/{len(ANOS)}')
for ano, caminho in sorted(arquivos.items()):
    tamanho_mb = caminho.stat().st_size / 1_000_000
    print(f'  {ano}: {caminho.name} ({tamanho_mb:.1f} MB)')

In [ ]:
# Célula 4 — Inspeção de colunas (confirma D-07)
from preprocessamento.filtrar_tematico import inspecionar_colunas

if arquivos:
    primeiro_csv = sorted(arquivos.values())[0]
    info_cols = inspecionar_colunas(primeiro_csv)
    print(f'Arquivo: {primeiro_csv.name}')
    print(f'Total de colunas: {len(info_cols)}')
    print('\nColunas de interesse:')
    colunas_interesse = ['NUMACORDAO', 'SITUACAO', 'ASSUNTO', 'SUMARIO', 'ACORDAO', 'VOTO']
    print(info_cols[info_cols['coluna'].isin(colunas_interesse)].to_string(index=False))

In [ ]:
# Célula 5 — Filtro temático + extração de label
from preprocessamento.filtrar_tematico import combinar_anos, salvar_parquet

PARQUET_FILTRADO = DATA_INTERIM / 'acordaos_filtrados.parquet'

df = combinar_anos(list(arquivos.keys()), data_dir=DATA_RAW)
salvar_parquet(df, PARQUET_FILTRADO)

print(f'\nDataset filtrado: {len(df)} acórdãos')
print('\nDistribuição de classes:')
print(df['LABEL'].value_counts())
print('\nDistribuição por ano:')
print(df.groupby(['ANO', 'LABEL']).size().unstack(fill_value=0))

In [ ]:
# Célula 6 — EDA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df = pd.read_parquet(PARQUET_FILTRADO)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribuição de classes
contagens = df['LABEL'].value_counts()
axes[0].bar(contagens.index, contagens.values, color=['#e74c3c','#f39c12','#27ae60'])
axes[0].set_title('Distribuição de Classes')
axes[0].set_ylabel('n acórdãos')
for i, v in enumerate(contagens.values):
    axes[0].text(i, v + 1, f'{v}\n({100*v/len(df):.1f}%)', ha='center', fontsize=9)

# Comprimento de SUMARIO (tokens aprox. por espaços)
df['len_sumario'] = df['SUMARIO'].fillna('').apply(lambda x: len(x.split()))
axes[1].hist(df['len_sumario'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Comprimento SUMARIO (tokens aprox.)')
axes[1].set_xlabel('Tokens')
axes[1].axvline(512, color='red', linestyle='--', label='Limite BERT')
axes[1].legend()

# Evolução temporal
temporal = df.groupby(['ANO', 'LABEL']).size().unstack(fill_value=0)
temporal.plot(kind='bar', ax=axes[2], color=['#e74c3c','#f39c12','#27ae60'])
axes[2].set_title('Acórdãos por Ano e Desfecho')
axes[2].set_xlabel('Ano')
axes[2].tick_params(axis='x', rotation=45)

fig.tight_layout()
fig.savefig(FIGURAS / 'eda_visao_geral.png', dpi=150)
plt.show()
print('Figura salva: eda_visao_geral.png')

In [ ]:
# Célula 7 — Limpeza e split estratificado 70/15/15
from preprocessamento.limpeza import aplicar_limpeza, dividir_dados, salvar_splits

# Limpeza para TF-IDF (modo tfidf = lowercase + stopwords)
print('Limpando SUMARIO para TF-IDF...')
df['SUMARIO_TFIDF'] = aplicar_limpeza(df, campo='SUMARIO', modo='tfidf')
print('Limpando VOTO para TF-IDF...')
df['VOTO_TFIDF'] = aplicar_limpeza(df, campo='VOTO', modo='tfidf')

# Limpeza para BERT (modo bert = preserva case + head+tail)
# Nota: head+tail requer download do tokenizer LegalBert-pt (~440 MB na 1ª execução)
print('Limpando SUMARIO para BERT (head+tail)...')
df['SUMARIO_BERT'] = aplicar_limpeza(df, campo='SUMARIO', modo='bert')
print('Limpando VOTO para BERT (head+tail)...')
df['VOTO_BERT'] = aplicar_limpeza(df, campo='VOTO', modo='bert')

# Split
train_df, val_df, test_df = dividir_dados(df)
salvar_splits(train_df, val_df, test_df, DATA_PROCESSED)

print(f'\nTreino: {len(train_df)} | Val: {len(val_df)} | Teste: {len(test_df)}')

In [ ]:
# Célula 8 — Baseline TF-IDF (ablação 2×2 — eixo TF-IDF)
from modelos.baseline import treinar_baseline_kfold, plotar_resultados

train_df = pd.read_parquet(DATA_PROCESSED / 'train.parquet')
resultados_baseline = {}

for campo_limpo, nome in [('SUMARIO_TFIDF', 'TF-IDF + LogReg | SUMARIO'),
                           ('VOTO_TFIDF',   'TF-IDF + LogReg | VOTO')]:
    print(f'\n=== {nome} ===')
    res = treinar_baseline_kfold(train_df[campo_limpo], train_df['LABEL'])
    resultados_baseline[nome] = res
    plotar_resultados(res, titulo=nome, output_dir=FIGURAS)
    print(f'F1-macro: {res["mean_f1"]:.4f} ± {res["std_f1"]:.4f} | IC95: {res["ci_95"]}')

print('\n=== Resumo Baseline ===')
for nome, res in resultados_baseline.items():
    print(f'{nome}: {res["mean_f1"]:.4f}')

In [ ]:
# Célula 9 — Fine-tuning LegalBert-pt com LoRA K-Fold (ablação 2×2 — eixo Transformer)
# ATENÇÃO: Requer GPU T4. Cada fold leva ~15–20 min. Total: ~2.5h por campo.
from modelos.transformer import kfold_com_lora

resultados_transformer = {}

for campo, nome in [('SUMARIO_BERT', 'LegalBert-pt | SUMARIO'),
                    ('VOTO_BERT',    'LegalBert-pt | VOTO')]:
    print(f'\n=== {nome} ===')
    res = kfold_com_lora(train_df, campo=campo)
    resultados_transformer[nome] = res
    print(f'F1-macro: {res["mean_f1"]:.4f} ± {res["std_f1"]:.4f} | IC95: {res["ci_95"]}')

In [ ]:
# Célula 10 — Comparação completa (Tabela 2×2)
from avaliacao.metricas import comparar_modelos

todos_resultados = {**resultados_baseline, **resultados_transformer}
tabela = comparar_modelos(todos_resultados)

print('\n=== Ablação 2×2 — Resultados Finais ===')
print(tabela.to_string(index=False))

tabela.to_csv(RESULTADOS / 'ablacao_2x2.csv', index=False)

In [ ]:
# Célula 11 — Threshold ótimo (custo assimétrico: FN=10×, FP=1×)
from avaliacao.metricas import otimizar_threshold
from modelos.baseline import construir_pipeline

test_df = pd.read_parquet(DATA_PROCESSED / 'test.parquet')

# Treina pipeline final no treino completo
pipeline_final = construir_pipeline('logreg')
pipeline_final.fit(train_df['SUMARIO_TFIDF'], train_df['LABEL'])

y_proba = pipeline_final.predict_proba(test_df['SUMARIO_TFIDF'])
classes = pipeline_final.classes_.tolist()
idx_irregular = classes.index('Irregular')
proba_irregular = y_proba[:, idx_irregular]

from modelos.transformer import LABEL2ID
y_true_ids = [LABEL2ID[l] for l in test_df['LABEL']]

threshold_otimo, metricas_thr = otimizar_threshold(
    y_true_ids, proba_irregular, custo_fn=10.0, custo_fp=1.0, output_dir=FIGURAS
)
print(f'\nThreshold ótimo: {threshold_otimo:.2f}')
print(f'F1-Irregular: {metricas_thr["f1_irregular"]:.4f}')
print(f'Recall-Irregular: {metricas_thr["recall_irregular"]:.4f}')

In [ ]:
# Célula 12 — Explicabilidade LIME (baseline)
from avaliacao.metricas import explicar_com_lime

textos_teste = test_df['SUMARIO_TFIDF'].tolist()
labels_teste = [LABEL2ID[l] for l in test_df['LABEL']]

# Garante pipeline com predict_proba treinado
explicar_com_lime(
    modelo=pipeline_final,
    textos=textos_teste,
    labels_verdadeiros=labels_teste,
    tokenizer=None,  # None = pipeline sklearn
    n_samples=5,
    output_dir=FIGURAS,
)
print('HTMLs LIME salvos em:', FIGURAS)

In [ ]:
# Célula 13 — Salvar todos os resultados em metricas.json
from avaliacao.metricas import salvar_metricas_json
import json

metricas_completas = {
    'corpus': {
        'total_acordaos': int(len(df)),
        'anos': ANOS,
        'distribuicao': df['LABEL'].value_counts().to_dict(),
    },
    'ablacao_2x2': {
        nome: {
            'mean_f1': res['mean_f1'],
            'std_f1': res['std_f1'],
            'ci_95': list(res['ci_95']),
            'mean_acc': res.get('mean_acc', 0),
        }
        for nome, res in todos_resultados.items()
    },
    'threshold_otimo': {
        'valor': threshold_otimo,
        'custo_fn': 10.0,
        'custo_fp': 1.0,
        **metricas_thr,
    },
}

salvar_metricas_json(metricas_completas, RESULTADOS / 'metricas.json')
print('Resultados salvos:', RESULTADOS / 'metricas.json')
print(json.dumps(metricas_completas, indent=2, default=str)[:500], '...')